In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DefaultDataCollator,
)
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
dataset = load_dataset("squad")
print(dataset)

In [ ]:
model_id = "model"
qa_model_id = "qa"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForQuestionAnswering.from_pretrained(
    model_id,
    ignore_mismatched_sizes=True,
)

model_size = sum(t.numel() for t in model.parameters())
print(f"Transformer size: {model_size/1000**2:.1f}M parameters")


In [ ]:
max_length = 256
doc_stride = 64

def tokenize(example):
    questions = [q.strip() for q in example["question"]]

    tokenized = tokenizer(
        questions,
        example["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized.sequence_ids(i)
        sample_idx = sample_map[i]
        answers = example["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

dataset.set_format("torch")
print(dataset)

In [ ]:
training_args = TrainingArguments(
    output_dir=qa_model_id,
    learning_rate=2e-5,
    weight_decay=0.02,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",

    logging_steps=500,
    eval_steps=500,
    save_steps=500,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    fp16=True,
    seed=42,
)

data_collator = DefaultDataCollator()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [ ]:
trainer.train()

In [ ]:
trainer.save_model(qa_model_id)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
logs = trainer.state.log_history

train_steps = [log["step"] for log in logs if "loss" in log and "eval_loss" not in log]
train_losses = [log["loss"] for log in logs if "loss" in log and "eval_loss" not in log]

eval_steps = [log["step"] for log in logs if "eval_loss" in log]
eval_losses = [log["eval_loss"] for log in logs if "eval_loss" in log]

plt.figure(figsize=(12, 6))
plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Validation Loss", linewidth=2)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

In [ ]:
metrics = trainer.evaluate(dataset["validation"])
print(metrics)

In [ ]:
from transformers import pipeline

qa = pipeline(
    "question-answering",
    model=qa_model_id,
    tokenizer=qa_model_id,
)

question = "What is the capital of France?"
context = "France is a country in Europe. Its capital is Paris."

result = qa(question=question, context=context)
print(f"Answer: {result['answer']} -> Score: {result['score']:.4f}")
print("-" * 60)